# PLN — Pré-processamento e Geração dos Dados

**Objetivo:** gerar o arquivo `entregas/p2/entrega_2.csv` a partir dos tweets brutos (`data/tweets.csv`), aplicando toda a pipeline de pré-processamento e vetorização TF-IDF.

Este notebook deve ser executado **antes** dos demais notebooks (`bow.ipynb`, `tdidf.ipynb`, etc.), pois eles consomem os arquivos gerados aqui.

### Pipeline (ordem correta)

1. **Remoção de URLs** — antes da tokenização, para evitar que `https://t.co/...` seja fragmentado
2. **Remoção de menções** (@user) — também antes da tokenização
3. **Remoção de emojis e decorações**
4. **Remoção de tokens puramente numéricos**
5. **Tokenização** com NLTK `TweetTokenizer`
6. **Remoção de stopwords** com spaCy (português)
7. **Stemming** com NLTK `SnowballStemmer('portuguese')`
8. **Normalização regex** (lowercase, hashtags, alfanuméricos, espaços)
9. **TF-IDF** com scikit-learn `TfidfVectorizer`
10. **Exportação** do CSV e JSON de metadados

### Correção aplicada

A pipeline original tokenizava o texto **antes** de remover URLs, fazendo com que `https://t.co/abc` fosse quebrado em fragmentos como `["https", "t", "co", "abc"]`. A regex de URL não conseguia mais identificá-los. Agora URLs são removidas no passo 1.

---

In [1]:
%pip install nltk spacy scikit-learn plotly seaborn matplotlib pandas numpy --quiet


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import csv
import json
import re
import sys
import os
from pathlib import Path

import nltk
from nltk.stem import SnowballStemmer
from nltk.tokenize import TweetTokenizer
from spacy.lang.pt.stop_words import STOP_WORDS as SPACY_PT_STOP_WORDS
from sklearn.feature_extraction.text import TfidfVectorizer

from tqdm.auto import tqdm

nltk.download('punkt_tab', quiet=True)

print('Dependencias carregadas.')

/Users/pedrohenriquewindisch/projects/edtwt/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dependencias carregadas.


## Definição das Expressões Regulares e Ferramentas NLP

In [3]:
TOKENIZADOR_NLTK = TweetTokenizer(preserve_case=True, reduce_len=False, strip_handles=False)
RADICALIZADOR_NLTK = SnowballStemmer("portuguese")

REGEX_TOKEN_COM_CONTEUDO = re.compile(r"\w", re.UNICODE)
REGEX_URL = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
REGEX_MENCAO = re.compile(r"@\w+", re.UNICODE)
REGEX_HASHTAG = re.compile(r"#(\w+)", re.UNICODE)
REGEX_NAO_ALFANUMERICO = re.compile(r"[^\w\s]", re.UNICODE)
REGEX_ESPACOS = re.compile(r"\s+", re.UNICODE)
TOKENS_NUMERICOS = re.compile(r"^\d+$", re.UNICODE)

TEXTO_COM_EMOJIS = re.compile(
    r"[^a-zA-ZÀ-ÿ0-9\s.,!?'\-"
    r"#@"
    r"\U0001F300-\U0001F5FF"
    r"\U0001F600-\U0001F64F"
    r"\U0001F680-\U0001F6FF"
    r"\U0001F700-\U0001F77F"
    r"\U0001F780-\U0001F7FF"
    r"\U0001F800-\U0001F8FF"
    r"\U0001F900-\U0001F9FF"
    r"\U0001FA00-\U0001FAFF"
    r"\U00002700-\U000027BF"
    r"\U00002600-\U000026FF"
    r"\u200d\uFE0F]"
)

print('Regex e ferramentas NLP definidas.')

Regex e ferramentas NLP definidas.


## Funções de Pré-processamento

Mesma lógica do `extrator/entrega.py`, com a diferença que URLs e menções são removidas **antes** da tokenização.

In [4]:
def normalizar_texto_csv(valor):
    if valor is None:
        return ""
    return str(valor)


def remover_decoracoes_com_regex(texto):
    return TEXTO_COM_EMOJIS.sub(" ", texto)


def remover_numericos_com_regex(texto):
    return " ".join(
        token
        for token in texto.split()
        if not TOKENS_NUMERICOS.match(token)
    )


def tokenizar_com_nltk(texto):
    return [
        token
        for token in TOKENIZADOR_NLTK.tokenize(texto)
        if REGEX_TOKEN_COM_CONTEUDO.search(token)
    ]


def remover_stopwords_com_spacy(tokens):
    return [
        token
        for token in tokens
        if token.casefold() not in SPACY_PT_STOP_WORDS
    ]


def aplicar_stemming_com_nltk(tokens):
    return " ".join(RADICALIZADOR_NLTK.stem(token.casefold()) for token in tokens)


def normalizar_com_regex(texto):
    texto_normalizado = texto.casefold()
    texto_normalizado = REGEX_URL.sub(" ", texto_normalizado)
    texto_normalizado = REGEX_HASHTAG.sub(r" \1 ", texto_normalizado)
    texto_normalizado = REGEX_MENCAO.sub(" ", texto_normalizado)
    texto_normalizado = REGEX_NAO_ALFANUMERICO.sub(" ", texto_normalizado)
    return REGEX_ESPACOS.sub(" ", texto_normalizado).strip()


def para_json(valor, ordenar_chaves=False):
    return json.dumps(valor, ensure_ascii=False, sort_keys=ordenar_chaves)


def processar_linha_preprocessamento(linha):
    """
    Pipeline completa de pre-processamento para um tweet.
    Ordem corrigida: URLs e mencoes removidas ANTES da tokenizacao.
    """
    texto = normalizar_texto_csv(linha.get("text"))

    # 1. Remover URLs (ANTES da tokenizacao)
    texto = REGEX_URL.sub(" ", texto)

    # 2. Remover mencoes (ANTES da tokenizacao)
    texto = REGEX_MENCAO.sub(" ", texto)

    # 3. Remover emojis e decoracoes
    texto = remover_decoracoes_com_regex(texto)

    # 4. Remover tokens puramente numericos
    texto = remover_numericos_com_regex(texto)

    # 5. Tokenizar
    tokens = tokenizar_com_nltk(texto)

    # 6. Remover stopwords
    tokens_sem_stopwords = remover_stopwords_com_spacy(tokens)

    # 7. Stemming
    texto_radicalizado = aplicar_stemming_com_nltk(tokens_sem_stopwords)

    # 8. Normalizacao regex
    texto_normalizado = normalizar_com_regex(texto_radicalizado)

    # Guardar resultados intermediarios
    linha["tokenizacao_nltk"] = para_json(tokens)
    linha["remocao_stopwords_spacy"] = para_json(tokens_sem_stopwords)
    linha["stemming_nltk"] = texto_radicalizado
    linha["normalizacao_re"] = texto_normalizado

    return texto_normalizado

print('Funcoes de pre-processamento definidas.')

Funcoes de pre-processamento definidas.


## Carregamento dos Tweets Brutos

In [5]:
CAMINHO_ENTRADA = Path('..') / 'data' / 'tweets.csv'

if not CAMINHO_ENTRADA.exists():
    raise FileNotFoundError(f'Arquivo nao encontrado: {CAMINHO_ENTRADA.resolve()}')

with CAMINHO_ENTRADA.open('r', newline='', encoding='utf-8') as f:
    leitor = csv.DictReader(f)
    linhas = list(leitor)
    cabecalhos = list(leitor.fieldnames or [])

print(f'Tweets carregados: {len(linhas)}')
print(f'Colunas originais: {len(cabecalhos)}')
print(f'Amostra (texto): {linhas[0].get("text", "")[:120]}...')

Tweets carregados: 2815
Colunas originais: 28
Amostra (texto): Refeições de hj! #wieiad ₍ᐢ. .ᐢ₎
．☆．。．:*･ﾟ

Gente eu esqueci de tirar foto do almoço, mas foi arroz com cenouras e um st...


## Execução da Pipeline de Pré-processamento

In [6]:
textos_normalizados = []

for indice, linha in enumerate(tqdm(linhas, desc='Pre-processando tweets'), start=1):
    texto_norm = processar_linha_preprocessamento(linha)
    textos_normalizados.append(texto_norm)

print(f'\nTweets pre-processados: {len(textos_normalizados)}')
print(f'Tweets com texto vazio apos processamento: {sum(1 for t in textos_normalizados if not t)}')

# Exemplo do primeiro tweet processado
print(f'\nExemplo:')
print(f'  Original:      {linhas[0].get("text", "")[:120]}')
print(f'  Tokens (NLTK): {linhas[0].get("tokenizacao_nltk", "")[:120]}')
print(f'  Stemmed:       {linhas[0].get("stemming_nltk", "")[:120]}')
print(f'  normalizacao_re: {linhas[0].get("normalizacao_re", "")[:120]}')

Pre-processando tweets: 100%|██████████| 2815/2815 [00:00<00:00, 6175.07it/s]


Tweets pre-processados: 2815
Tweets com texto vazio apos processamento: 0

Exemplo:
  Original:      Refeições de hj! #wieiad ₍ᐢ. .ᐢ₎
．☆．。．:*･ﾟ

Gente eu esqueci de tirar foto do almoço, mas foi arroz com cenouras e um st
  Tokens (NLTK): ["Refeições", "de", "hj", "#wieiad", "Gente", "eu", "esqueci", "de", "tirar", "foto", "do", "almoço", "mas", "foi", "arr
  Stemmed:       refeiçõ hj #wieiad gent esquec tir fot almoc arroz cenour steak frang 1.244 kcals 43.2 g prot gast klcals caminh muscul 
  normalizacao_re: refeiçõ hj wieiad gent esquec tir fot almoc arroz cenour steak frang 1 244 kcals 43 2 g prot gast klcals caminh muscul m


## TF-IDF com scikit-learn

Mesmos hiperparâmetros da Entrega 2 original.

In [7]:
vetorizador = TfidfVectorizer(
    lowercase=False,
    max_df=0.85,
    min_df=2,
    max_features=1000,
    token_pattern=r"(?u)\b\w\w+\b",
)

# Substituir textos vazios por placeholder para evitar erro
textos_para_vetorizar = [t if t else ' ' for t in textos_normalizados]

# Filtrar linhas validas para o TfidfVectorizer
if any(textos_para_vetorizar):
    matriz = vetorizador.fit_transform(textos_para_vetorizar)
    nomes_features = vetorizador.get_feature_names_out().tolist()
else:
    matriz = None
    nomes_features = []

print(f'Features TF-IDF: {len(nomes_features)} termos')
print(f'Matriz TF-IDF: {matriz.shape if matriz is not None else "vazia"}')
print(f'\nTermos mais frequentes (soma TF-IDF):')
if matriz is not None:
    import numpy as np
    soma_tfidf = np.array(matriz.sum(axis=0)).flatten()
    top_idx = np.argsort(soma_tfidf)[-15:][::-1]
    for idx in top_idx:
        print(f'  {nomes_features[idx]:20s} {soma_tfidf[idx]:.2f}')

Features TF-IDF: 1000 termos
Matriz TF-IDF: (2815, 1000)

Termos mais frequentes (soma TF-IDF):
  com                  87.03
  pra                  74.39
  vou                  56.42
  hoj                  53.04
  almoc                51.57
  pro                  50.85
  thread               50.34
  recovery             47.79
  wieiad               41.59
  fic                  40.29
  ed                   40.23
  to                   38.78
  vcs                  37.16
  dia                  36.43
  calor                35.93


In [8]:
features_por_linha = [para_json({}) for _ in linhas]

if matriz is not None:
    for indice, vetor_linha in enumerate(tqdm(matriz, desc='Extraindo features TF-IDF'), start=1):
        mapa_features = {
            nomes_features[indice_coluna]: round(float(valor), 6)
            for indice_coluna, valor in zip(vetor_linha.indices, vetor_linha.data)
        }
        features_por_linha[indice - 1] = para_json(mapa_features, ordenar_chaves=True)

for linha, feat_json in zip(linhas, features_por_linha):
    linha["features_tfidf_sklearn"] = feat_json

print('Features TF-IDF salvas nas linhas.')

Extraindo features TF-IDF: 2815it [00:00, 55806.69it/s]

Features TF-IDF salvas nas linhas.


## Exportação: `entrega_2.csv` e `entrega_2_tfidf_features.json`

In [9]:
PASTA_SAIDA = Path('..') / 'entregas' / 'p2'
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

caminho_csv = PASTA_SAIDA / 'entrega_2.csv'
caminho_json = PASTA_SAIDA / 'entrega_2_tfidf_features.json'

# Salvar entrega_1.csv tambem (para compatibilidade com entrega_2.py original)
PASTA_ENTREGA_1 = Path('..') / 'entregas' / 'p1'
PASTA_ENTREGA_1.mkdir(parents=True, exist_ok=True)
caminho_csv_p1 = PASTA_ENTREGA_1 / 'entrega_1.csv'

In [10]:
novas_colunas = [
    "tokenizacao_nltk",
    "remocao_stopwords_spacy",
    "stemming_nltk",
    "normalizacao_re",
    "features_tfidf_sklearn",
]

cabecalhos_saida = [*cabecalhos, *novas_colunas]

with caminho_csv.open('w', newline='', encoding='utf-8') as f:
    escritor = csv.DictWriter(f, fieldnames=cabecalhos_saida)
    escritor.writeheader()
    escritor.writerows(linhas)

print(f'entrega_2.csv salvo: {caminho_csv.resolve()}')
print(f'  Linhas: {len(linhas)}')
print(f'  Colunas: {len(cabecalhos_saida)}')

entrega_2.csv salvo: /Users/pedrohenriquewindisch/projects/edtwt/entregas/p2/entrega_2.csv
  Linhas: 2815
  Colunas: 33


In [11]:
params = vetorizador.get_params()
metadados = {
    "feature_names": nomes_features,
    "vectorizer": {
        "max_df": params["max_df"],
        "min_df": params["min_df"],
        "max_features": params["max_features"],
        "token_pattern": params["token_pattern"],
        "lowercase": params["lowercase"],
    },
}

caminho_json.write_text(
    json.dumps(metadados, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

print(f'entrega_2_tfidf_features.json salvo: {caminho_json.resolve()}')
print(f'  Features: {len(nomes_features)}')

entrega_2_tfidf_features.json salvo: /Users/pedrohenriquewindisch/projects/edtwt/entregas/p2/entrega_2_tfidf_features.json
  Features: 1000


In [12]:
# Salvar tambem entrega_1.csv para compatibilidade com o pipeline original
colunas_p1 = [c for c in cabecalhos_saida if c not in ("normalizacao_re", "features_tfidf_sklearn")]
with caminho_csv_p1.open('w', newline='', encoding='utf-8') as f:
    escritor = csv.DictWriter(f, fieldnames=colunas_p1)
    escritor.writeheader()
    escritor.writerows(linhas)

print(f'entrega_1.csv salvo: {caminho_csv_p1.resolve()}')
print(f'  Linhas: {len(linhas)}')
print(f'  Colunas: {len(colunas_p1)}')

ValueError: dict contains fields not in fieldnames: 'normalizacao_re', 'features_tfidf_sklearn'

## Verificação: Termos Mais Frequentes

O vocabulário não deve conter fragmentos de URL como `https` ou `co`.

In [ ]:
import numpy as np

print('Top 15 termos por soma TF-IDF (nao devem conter https, co, t, etc.):')
soma_tfidf = np.array(matriz.sum(axis=0)).flatten()
top_idx = np.argsort(soma_tfidf)[-15:][::-1]
for idx in top_idx:
    print(f'  {nomes_features[idx]:20s} {soma_tfidf[idx]:.2f}')

# Verificar se https esta no vocabulario
termos_suspeitos = ['https', 'co', 'www', 'http', 'com', 't']
presentes = [t for t in termos_suspeitos if t in nomes_features]
if presentes:
    print(f'\nATENCAO: Termos suspeitos ainda presentes no vocabulario: {presentes}')
else:
    print(f'\nOK: Nenhum termo suspeito de fragmento de URL encontrado.')

## Próximos Passos

Após executar este notebook, os seguintes arquivos estarão disponíveis:

- `entregas/p2/entrega_2.csv` — tweets com colunas de pré-processamento e TF-IDF
- `entregas/p2/entrega_2_tfidf_features.json` — metadados do TfidfVectorizer
- `entregas/p1/entrega_1.csv` — tweets com colunas de pré-processamento (sem TF-IDF)

Agora você pode executar os notebooks de análise:
- `bow.ipynb` — Bag of Words
- `tdidf.ipynb` — TF-IDF (features pré-computadas)
- `tdidf-lematizacao-stemming.ipynb` — TF-IDF com lematização + stemming
- `word2vec.ipynb` — Word2Vec
- `bert.ipynb` — BERT

---

*Nota: Os notebooks `word2vec.ipynb` e `bert.ipynb` dependem das entregas p3 e p4, que devem ser geradas pelo pipeline principal do projeto (`processadores/`).*